## **First RAG Pipeline Code**

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("../../data/static/parking_info.txt")

documents = loader.load()

print(documents[0].page_content)

Welcome to Smart Parking Center.

Location:
45 Downtown Avenue, Tech City.

Working Hours:
Open 24/7.

Parking Types:
- Standard Parking
- VIP Parking
- Electric Vehicle Charging Slots

Reservation Policy:
Reservations can be made up to 30 days in advance.

Cancellation Policy:
Reservations may be cancelled up to 2 hours before booking time.

Security:
24/7 CCTV surveillance and security staff available.

Pricing:
Standard Parking: $5/hour
VIP Parking: $10/hour
EV Charging Slot: $8/hour

Booking Process:
Users must provide:
- First Name
- Last Name
- Car Number
- Reservation Start Date
- Reservation End Date

Contact:
support@smartparking.ai


## **Chunking**

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 3


## **Create Embeddings**

In [7]:


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_2456\353082983.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\ShoaibMohdKhan\Desktop\parking-rag-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ShoaibMohdKhan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## **Create Chroma Vector DB**

In [8]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../../data/chroma_db"
)

vector_db.persist()

print("Vector database created successfully.")

Vector database created successfully.


C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_2456\568718003.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


## **Test Retrieval**

In [9]:
query = "What are the parking prices?"

results = vector_db.similarity_search(query, k=2)

for idx, result in enumerate(results):
    print(f"\nResult {idx+1}:\n")
    print(result.page_content)


Result 1:

Cancellation Policy:
Reservations may be cancelled up to 2 hours before booking time.

Security:
24/7 CCTV surveillance and security staff available.

Pricing:
Standard Parking: $5/hour
VIP Parking: $10/hour
EV Charging Slot: $8/hour

Result 2:

Welcome to Smart Parking Center.

Location:
45 Downtown Avenue, Tech City.

Working Hours:
Open 24/7.

Parking Types:
- Standard Parking
- VIP Parking
- Electric Vehicle Charging Slots

Reservation Policy:
Reservations can be made up to 30 days in advance.


## **Add Dynamic SQL-like Data Loading**

In [10]:
import pandas as pd

dynamic_data = pd.read_csv("../../data/dynamic/parking_dynamic.csv")

dynamic_data

,slot_type,available_slots,last_updated
0,Standard,48,2026-05-26 10:00
1,VIP,7,2026-05-26 10:00
2,EV,3,2026-05-26 10:00


## **Create First Retrieval Function**

In [11]:
def retrieve_parking_info(query: str):

    results = vector_db.similarity_search(query, k=2)

    context = "\n\n".join(
        [doc.page_content for doc in results]
    )

    return context

## **Test Retrieval Function**

In [12]:
response = retrieve_parking_info(
    "Tell me about parking pricing"
)

print(response)

Cancellation Policy:
Reservations may be cancelled up to 2 hours before booking time.

Security:
24/7 CCTV surveillance and security staff available.

Pricing:
Standard Parking: $5/hour
VIP Parking: $10/hour
EV Charging Slot: $8/hour

Welcome to Smart Parking Center.

Location:
45 Downtown Avenue, Tech City.

Working Hours:
Open 24/7.

Parking Types:
- Standard Parking
- VIP Parking
- Electric Vehicle Charging Slots

Reservation Policy:
Reservations can be made up to 30 days in advance.
